### 1. Definizione e Orchestrazione degli Esperimenti
Questa cella definisce le funzioni principali per caricare i dati delle feature con contesto, addestrare modelli (RandomForest, SVM, XGBoost) sulle varie fold, e salvare i risultati dei test. Contiene inoltre la logica per analizzare i risultati e addestrare il modello finale migliore sull'intero dataset.

In [ ]:
import os
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        confusion_matrix,
        classification_report
    )

from xgboost import XGBClassifier
from pathlib import Path
from google.colab import drive
import joblib # Added for model serialization

# Mount Google Drive
drive.mount('/content/drive')

# Configuration
BASE_PATH = "/content/drive/MyDrive/HAR"
# Pointing back to with_context folder to have access to all columns
DATA_PATH = Path(f"{BASE_PATH}/features_with_context")
RESULTS_BASE_DIR = Path(f"{BASE_PATH}/results/bodypart_context_experiments")

ACTIVITIES = [
    "lying",
    "running",
    "sitting",
    "standing",
    "walking"
]

LABEL_MAP = {act: i for i, act in enumerate(ACTIVITIES)}
INV_LABEL_MAP = {i: act for i, act in enumerate(ACTIVITIES)}

def load_data_from_dir(directory):
    dfs = []
    if not os.path.exists(directory):
        return pd.DataFrame()
    for file in os.listdir(directory):
        if file.endswith('.csv'):
            df = pd.read_csv(os.path.join(directory, file))
            dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def prepare_data(df):
    X = df.drop(columns=['activity', 'window_start', 'window_end'], errors='ignore')
    y = df['activity']
    X = X.replace([np.inf, -np.inf], np.nan)
    y_numeric = y.map(LABEL_MAP)
    return X, y_numeric

def get_metrics(y_true, y_pred, model_name, fold_idx):
    acc = accuracy_score(y_true, y_pred)
    prec_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    res = {'fold': fold_idx, 'model': model_name, 'accuracy': acc, 'precision_macro': prec_macro, 'recall_macro': rec_macro, 'f1_macro': f1_macro}
    report = classification_report(y_true, y_pred, target_names=ACTIVITIES, output_dict=True, zero_division=0)
    for act in ACTIVITIES:
        res[f'precision_{act}'] = report[act]['precision']
        res[f'recall_{act}'] = report[act]['recall']
        res[f'f1_{act}'] = report[act]['f1-score']
    return res

def save_fold_results(fold_name, model_name, y_true, y_pred, metrics_dict, base_output_dir):
    dest_path = base_output_dir / fold_name / model_name
    dest_path.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([metrics_dict]).to_csv(dest_path / 'metrics.csv', index=False)
    cm = confusion_matrix(y_true, y_pred, labels=range(len(ACTIVITIES)))
    pd.DataFrame(cm, index=ACTIVITIES, columns=ACTIVITIES).to_csv(dest_path / 'confusion_matrix.csv')
    preds_df = pd.DataFrame({'true_activity': [INV_LABEL_MAP[v] for v in y_true], 'predicted_activity': [INV_LABEL_MAP[v] for v in y_pred]})
    preds_df.to_csv(dest_path / 'predictions.csv', index=False)
    return preds_df

Mounted at /content/drive


In [ ]:
BODY_PARTS = ["chest", "forearm", "head", "shin", "thigh", "upperarm", "waist"]
CONTEXT_SENSORS = ["gps", "light", "mic", "all"]

def run_body_part_experiment(
    body_parts_to_include,
    context_sensor_to_include,
    experiment_tag,
    base_path_for_data,
    base_path_for_results
):
    current_experiment_results_dir = base_path_for_results / experiment_tag
    current_experiment_results_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n--- Running: {experiment_tag} | Body: {body_parts_to_include} | Context: {context_sensor_to_include} ---")

    all_fold_metrics = []
    global_predictions = {name: [] for name in ['RandomForest', 'SVM', 'XGBoost']}

    for i in range(1, 16):
        fold_name = f"folder{i}"
        train_df = load_data_from_dir(base_path_for_data / fold_name / 'train')
        test_df = load_data_from_dir(base_path_for_data / fold_name / 'test')

        if train_df.empty or test_df.empty: continue

        X_train_raw, y_train = prepare_data(train_df)
        X_test_raw, y_test = prepare_data(test_df)

        # Filtering Logic
        selected_prefixes = [f"{bp}_" for bp in body_parts_to_include]

        if context_sensor_to_include == "all":
            selected_prefixes.extend(["gps_", "light_", "mic_"])
        elif context_sensor_to_include:
            selected_prefixes.append(f"{context_sensor_to_include}_")

        cols = [col for col in X_train_raw.columns if any(col.startswith(p) for p in selected_prefixes)]
        X_train, X_test = X_train_raw[cols], X_test_raw[cols]

        if X_train.empty: continue

        models = {
            "RandomForest": Pipeline([("imputer", SimpleImputer(strategy="median")), ("classifier", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))]),
            "SVM": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("classifier", SVC(random_state=42))]),
            "XGBoost": Pipeline([("imputer", SimpleImputer(strategy="median")), ("classifier", XGBClassifier(random_state=42, eval_metric="mlogloss", n_jobs=-1))])
        }

        for name, model in models.items():
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            m = get_metrics(y_test, y_pred, name, i)
            all_fold_metrics.append(m)
            preds_df = save_fold_results(fold_name, name, y_test, y_pred, m, current_experiment_results_dir)
            global_predictions[name].append(preds_df)

    if not all_fold_metrics: return None, None

    fold_results_df = pd.DataFrame(all_fold_metrics)
    summary = fold_results_df.groupby('model')[['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']].agg(['mean', 'std']).reset_index()
    summary.columns = ['model'] + [f"{c[0]}_{c[1]}" for c in summary.columns[1:]]
    summary.to_csv(current_experiment_results_dir / 'summary.csv', index=False)

    return fold_results_df, summary

def get_model_config_from_tag(experiment_tag):
    """Parses the experiment tag to extract body parts and context sensor configuration."""
    parts_tag, sensor_tag = experiment_tag.rsplit('_plus_', 1)

    if parts_tag.startswith('only_'):
        body_parts = [parts_tag.replace('only_', '')]
    elif parts_tag.startswith('first_') and parts_tag.endswith('_parts'):
        num_parts = int(parts_tag.replace('first_', '').replace('_parts', ''))
        body_parts = BODY_PARTS[:num_parts]
    elif parts_tag == 'baseline_full_context': # Added for the new tag
        body_parts = BODY_PARTS
    else:
        raise ValueError(f"Unknown parts tag format: {parts_tag}")

    return body_parts, sensor_tag

def train_and_save_final_best_model(
    model_type,
    body_parts_to_include,
    context_sensor_to_include,
    base_path_for_data,
    output_dir
):
    print(f"\n--- Retraining and saving best {model_type} model for config: Body: {body_parts_to_include} | Context: {context_sensor_to_include} ---")

    all_train_dfs = []
    for i in range(1, 16):
        fold_name = f"folder{i}"
        train_df_fold = load_data_from_dir(base_path_for_data / fold_name / 'train')
        if not train_df_fold.empty:
            all_train_dfs.append(train_df_fold)

    if not all_train_dfs:
        print(f"No training data found for {model_type} in specified configuration.")
        return

    full_train_df = pd.concat(all_train_dfs, ignore_index=True)
    X_train_raw, y_train = prepare_data(full_train_df)

    selected_prefixes = [f"{bp}_" for bp in body_parts_to_include]
    if context_sensor_to_include == "all":
        selected_prefixes.extend(["gps_", "light_", "mic_"])
    elif context_sensor_to_include:
        selected_prefixes.append(f"{context_sensor_to_include}_")

    cols = [col for col in X_train_raw.columns if any(col.startswith(p) for p in selected_prefixes)]
    X_train = X_train_raw[cols]

    if X_train.empty:
        print(f"No features selected for {model_type} in specified configuration.")
        return

    if model_type == "RandomForest":
        model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("classifier", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))])
    elif model_type == "SVM":
        model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("classifier", SVC(random_state=42))])
    elif model_type == "XGBoost":
        model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("classifier", XGBClassifier(random_state=42, eval_metric="mlogloss", n_jobs=-1))])
    else:
        raise ValueError(f"Unknown model type: {model_type}")

    model.fit(X_train, y_train)
    model_save_path = output_dir / f'best_{model_type}_model.joblib'
    joblib.dump(model, model_save_path)
    print(f"Best {model_type} model saved to {model_save_path}")

def find_and_save_best_models(final_summary_df, results_base_dir, model_names, data_base_path):
    print("\n--- Finding and saving best models ---")
    best_models_summary = []
    for model_name in model_names:
        model_summary = final_summary_df[final_summary_df['model'] == model_name]
        if not model_summary.empty:
            # Get the experiment with the highest mean accuracy for the current model
            best_exp_for_model = model_summary.loc[model_summary['accuracy_mean'].idxmax()]
            best_models_summary.append(best_exp_for_model)

            best_experiment_tag = best_exp_for_model['experiment']
            print(f"Best experiment for {model_name}: {best_experiment_tag} (Accuracy: {best_exp_for_model['accuracy_mean']:.4f})")

            # Save individual best model summary
            pd.DataFrame([best_exp_for_model]).to_csv(results_base_dir / f'best_{model_name}_summary.csv', index=False)

            # Extract configuration and train/save the actual model
            body_parts, context_sensor = get_model_config_from_tag(best_experiment_tag)
            train_and_save_final_best_model(model_name, body_parts, context_sensor, data_base_path, results_base_dir)

        else:
            print(f"No results found for {model_name}.")

    if best_models_summary:
        pd.DataFrame(best_models_summary).to_csv(results_base_dir / 'best_overall_models_summary.csv', index=False)
    print("--- Best models saved ---")

def main_orchestrator():
    all_summaries = []
    results_dir = Path(f"{BASE_PATH}/results/bodypart_context_experiments")
    overall_summary_path = results_dir / 'overall_summary.csv'

    if overall_summary_path.exists():
        final_summary = pd.read_csv(overall_summary_path)
        print(f"Loaded existing overall summary from {overall_summary_path}")
    else:
        print(f"No existing overall summary found at {overall_summary_path}. Please run the experiments first or provide the file.")
        return

    if not final_summary.empty:
        display(final_summary)

        # Find and save the best performing experiment for each model
        model_names = ['RandomForest', 'SVM', 'XGBoost']
        find_and_save_best_models(final_summary, results_dir, model_names, DATA_PATH)
    else:
        print("Loaded summary is empty. No experiments were successfully completed or recorded.")

if __name__ == "__main__":
    main_orchestrator()

### 2. Identificazione e Salvataggio dei Modelli Migliori per Fold
Questa cella analizza il file di riepilogo `fold_results.csv` per identificare la fold migliore (con accuratezza più alta) per ciascun modello, carica i relativi dati di addestramento e salva i modelli addestrati in formato serializzato (`.joblib`).

In [ ]:
import pandas as pd
import joblib

from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier


# Updated BASE_PATH to point to the Drive directory where files are located
BASE_PATH = "/content/drive/MyDrive/HAR"
DATA_PATH = Path(f"{BASE_PATH}/features_with_context")

MODEL_NAMES = [
    "RandomForest",
    "SVM",
    "XGBoost"
]


def get_model(model_type):

    if model_type == "RandomForest":

        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", RandomForestClassifier(
                n_estimators=100,
                random_state=42,
                n_jobs=-1
            ))
        ])

    elif model_type == "SVM":

        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("classifier", SVC(
                random_state=42
            ))
        ])

    elif model_type == "XGBoost":

        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", XGBClassifier(
                random_state=42,
                eval_metric="mlogloss",
                n_jobs=-1
            ))
        ])

    else:
        raise ValueError(f"Unknown model type: {model_type}")


def load_data_from_dir(directory):
    directory = Path(directory)
    csv_files = sorted(directory.glob("*.csv"))

    if not csv_files:
        raise ValueError(
            f"No CSV files found in {directory}"
        )

    dfs = []

    for csv_file in csv_files:

        print(f"Loading: {csv_file}")

        df = pd.read_csv(csv_file)
        dfs.append(df)

    return pd.concat(
        dfs,
        ignore_index=True
    )


def prepare_data(df):
    # Re-using the logic from cell 4be41115 for consistency
    ACTIVITIES = ["lying", "running", "sitting", "standing", "walking"]
    LABEL_MAP = {act: i for i, act in enumerate(ACTIVITIES)}

    X = df.drop(columns=['activity', 'window_start', 'window_end'], errors='ignore')
    y = df['activity']
    X = X.replace([float('inf'), float('-inf')], float('nan'))
    y_numeric = y.map(LABEL_MAP)

    return X, y_numeric


def find_best_models():

    fold_results_path = Path(f"{BASE_PATH}/fold_results.csv")

    print(f"\nReading: {fold_results_path}")

    results = pd.read_csv(fold_results_path)

    best_models = []

    for model_name in MODEL_NAMES:

        model_results = results[
            results["model"] == model_name
        ]

        if model_results.empty:
            print(f"No results found for {model_name}")
            continue

        # Select the best fold for this model
        best_row = model_results.loc[
            model_results["accuracy"].idxmax()
        ]

        best_models.append(best_row)

    if not best_models:
        raise ValueError(
            "No valid model results found."
        )

    return pd.DataFrame(best_models)


def train_and_save_model(
    model_type,
    fold,
    accuracy,
    data_base_path,
    output_dir
):

    fold_name = f"folder{int(fold)}"

    train_path = (
        data_base_path /
        fold_name /
        "train"
    )

    if not train_path.exists():
        raise ValueError(
            f"Training directory not found: {train_path}"
        )

    print(
        f"\nLoading training data from: "
        f"{train_path}"
    )

    train_df = load_data_from_dir(
        train_path
    )

    if train_df.empty:
        raise ValueError(
            f"No training data found in {train_path}"
        )

    # All available features are used.
    X_train, y_train = prepare_data(
        train_df
    )

    print(
        f"Training samples: {len(X_train)}"
    )

    print(
        f"Features: {X_train.shape[1]}"
    )

    # Create model
    model = get_model(model_type)

    # Train
    model.fit(
        X_train,
        y_train
    )

    # Create output directory
    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # Save only the model
    output_path = (
        output_dir /
        f"best_{model_type}.joblib"
    )

    joblib.dump(
        model,
        output_path
    )

    print(
        f"\n{model_type}"
    )

    print(
        f"Best fold: {fold_name}"
    )

    print(
        f"Accuracy: {accuracy:.4f}"
    )

    print(
        f"Number of features: "
        f"{X_train.shape[1]}"
    )

    print(
        f"Model saved to: {output_path}"
    )


def main():

    data_base_path = Path(
        DATA_PATH
    )

    output_dir = Path(
        f"{BASE_PATH}/embedded_models"
    )

    best_models = find_best_models()

    print(
        "\n========================================"
    )
    print(
        "BEST MODELS"
    )
    print(
        "========================================"
    )

    print(
        best_models[
            [
                "model",
                "fold",
                "accuracy",
                "f1_macro"
            ]
        ].to_string(index=False)
    )

    for _, row in best_models.iterrows():

        train_and_save_model(
            model_type=row["model"],
            fold=row["fold"],
            accuracy=row["accuracy"],
            data_base_path=data_base_path,
            output_dir=output_dir
        )


if __name__ == "__main__":
    main()


Reading: /content/drive/MyDrive/HAR/fold_results.csv

BEST MODELS
       model  fold  accuracy  f1_macro
RandomForest    13  0.985095  0.985148
         SVM     2  0.994822  0.991593
     XGBoost    12  0.993214  0.993230

Loading training data from: /content/drive/MyDrive/HAR/features_with_context/folder13/train
Loading: /content/drive/MyDrive/HAR/features_with_context/folder13/train/dataset10_features.csv
Loading: /content/drive/MyDrive/HAR/features_with_context/folder13/train/dataset11_features.csv


KeyboardInterrupt: 

### 3. Esportazione dei Dati di Test in Formato C Header (`.h`)
Questa cella converte le feature di test in valori interi a 16 bit (`int16_t`) e genera un file header C (`test_data.h`) contenente la matrice dei dati per scopi di sviluppo embedded.

In [1]:
X_int16 = X_test.to_numpy().astype(np.int16)

with open("test_data.h", "w") as f:
    f.write("#ifndef TEST_DATA_H\n")
    f.write("#define TEST_DATA_H\n\n")
    f.write("#include <stdint.h>\n\n")

    f.write(f"#define NUM_SAMPLES {X_int16.shape[0]}\n")
    f.write(f"#define NUM_FEATURES {X_int16.shape[1]}\n\n")

    f.write(
        "static const int16_t test_data"
        "[NUM_SAMPLES][NUM_FEATURES] = {\n"
    )

    for row in X_int16:
        values = ", ".join(str(int(x)) for x in row)
        f.write(f"    {{{values}}},\n")

    f.write("};\n\n")
    f.write("#endif\n")

print("Saved test_data.h")
print("Shape:", X_int16.shape)


NameError: name 'X_test' is not defined

### 4. Verifica dell'Impatto della Quantizzazione (Int16) sulle Predizioni
Questa cella confronta le predizioni del modello originale (eseguite con dati in virgola mobile) con quelle eseguite convertendo i dati in `int16_t`, al fine di valutare se il processo di quantizzazione altera i risultati della classificazione.

In [ ]:
X_int16 = X_test.to_numpy().astype(np.int16)

pred_original = model.predict(X_test)
pred_int16 = model.predict(X_int16)

for i, (p1, p2) in enumerate(zip(pred_original, pred_int16)):
    print(
        f"Sample {i}: "
        f"original={p1}, "
        f"int16={p2}, "
        f"{'OK' if p1 == p2 else 'DIFFERENT'}"
    )